# 让网页真的会分析文字

## 动手换芯

打开 main.py，在文件顶部导入两个库：

~~~python
from pypinyin import lazy_pinyin, Style
from snownlp import SnowNLP
~~~

先把 analyze 改成：

~~~python
@app.post("/api/analyze")
def analyze(req: AnalyzeRequest):
    text = req.text
    score = round(SnowNLP(text).sentiments, 2)
    return {
        "text": text,
        "score": score,
        "label": "偏平静",
        "pinyin": " ".join(lazy_pinyin(text, style=Style.TONE)),
    }
~~~

主要变化有两处：

1. 用 SnowNLP 计算 score，并把结果放进返回值。
2. 用 lazy_pinyin 计算带声调的拼音，并把结果放进返回值。

## 把分数翻译成标签

把“数字 → 结论”的逻辑单独写成一个小函数，放在 analyze 上面：

~~~python
def score_label(score):
    if score >= 0.6:
        return "偏积极"
    elif score <= 0.4:
        return "偏消极"
    else:
        return "中性"
~~~

然后把 analyze 中的占位标签替换为 score_label(score)：

~~~python
@app.post("/api/analyze")
def analyze(req: AnalyzeRequest):
    text = req.text
    score = round(SnowNLP(text).sentiments, 2)

    return {
        "text": text,
        "score": score,
        "label": score_label(score),
        "pinyin": " ".join(lazy_pinyin(text, style=Style.TONE)),
    }
~~~

## 见证效果

修改后分别启动前端和后端。

前端终端：

~~~bash
npm run dev
~~~

后端终端：

~~~bash
uv run fastapi dev
~~~

打开 http://localhost:3000，进入文字实验室，输入一句话并点击“开始分析”。

现在应该看到：

- 拼音真正生成了，并且带着声调。
- 多音字会根据词语判断，例如“重庆”和“重要”中的“重”读音不同。
- 逗号等非汉字字符会原样保留，因为 lazy_pinyin 只转换汉字。

试试几组输入：

| 输入 | score | label |
| --- | ---: | --- |
| 我特别喜欢这部电影 | 0.95 | 偏积极 |
| 今天的风很轻，适合把想法写下来 | 0.95 | 偏积极 |
| 太失望了，再也不来了 | 0.00 | 偏消极 |